### Lists - test update dataResource in collectory 





Import libraries

In [ ]:
import json
import logging
import requests
import pandas as pd
from pathlib import Path
# 
cwd = Path.cwd()
dataDir = cwd.parent /  Path('data')
colfile = dataDir / Path(f'lists_1.csv')    # collectory metadata for DRs

collectory_url = "https://collections-test.ala.org.au/ws/"
collectory_api_key = "<preingestion baseconfig api key here>"

mdata = {"name": "00-dr22810 Rose Test - DR-List-updated-12", "isPrivate": "False"}
# mdata = {"name": "00-dr22810 Rose Test - DR-List-updated-11", "isPrivate": "false"}
# mdata = {"name": "00-dr22810 Rose Test - DR-List-updated-10", "isPrivate": False}
# mdata = {"name": "00-dr22810 Rose Test - DR-List-updated-09", "isPrivate": false}

In [9]:
# Functions

def join_url(*url_fragments: str) -> str:
        """
        Joins multiple URL fragments into a single URL.

        :param url_fragments: URL fragments to be joined.
        :return: A single URL string.
        """
        return "/".join(fragment.strip("/") for fragment in url_fragments)

def json_parse(base_url: str, url_path: str, params=None, headers=None, method="GET"
):
    """
    Calls the specified URL and returns the JSON response.
    :param base_url: like https://collections.ala.org.au/ws
    :param url_path: like /dataResource/dr000
    :param params: is a dictionary of parameters to be passed to the API
    :return: is the json response from the URL
    """

    try:
        full_url = join_url(base_url, url_path)
        if method == "GET":
            with requests.get(
                full_url, params, headers=headers, timeout=60
            ) as response:
                response.raise_for_status()
                json_result = json.loads(response.content)
                return json_result
        elif method == "POST":
            with requests.post(
                full_url, json=params, headers=headers, timeout=60
            ) as response:
                response.raise_for_status()
                return response.content
    except requests.exceptions.HTTPError as err:
            # logging.error(
            logging.error(
                "Error encountered during request %s with params %s",
                full_url,
                params,
                exc_info=err,
            )
            raise IOError(err)

def update_registry_metadata(registry_base_url, uid, ala_api_key, metadata):
    """
    Updates metadata for a dataset in the registry (Collectory) using the API key.

    Args:
        registry_base_url (str): The base URL of the registry (Collectory).
        uid (str): The unique identifier of the dataset.
        ala_api_key (str): The API key for authentication.
        metadata (dict): The metadata to update.

    Returns:
        dict: The updated metadata of the dataset, or an error message if not found.
    """
    if uid.startswith("dr"):
        resource_path = f"dataResource/{uid}"
    elif uid.startswith("dp"):
        resource_path = f"dataProvider/{uid}"
    else:
        raise ValueError("Not a valid dataset or data provider uid: %s", uid)

    try:
        jresponse = json_parse(
            registry_base_url,
            resource_path,
            headers={"Authorization": ala_api_key},
            # method="GET",
            method="POST",
            params=metadata,
        )
        return jresponse
    except Exception as e:
        print(f"Error fetching metadata for {uid}: {str(e)}")


In [31]:
coll_df = pd.read_csv(colfile, encoding="utf-8", dtype="str").fillna(
    ""
)
# metadf = coll_df.drop(columns=["id", "version"])
dict_list = json.loads(
    coll_df.to_json(orient="records", double_precision=False)
)
metadf = coll_df
metadf["json_string"] = ""
metadf["json_string"] = [json.dumps(item) for item in dict_list]
# metadf["isPrivate"] = False
# metadf["isPrivate"] = True
druid = metadf.loc[0, "dataResourceUid"]
# mdata = metadf.loc[0, "json_string"]
print(mdata)
new_response = update_registry_metadata(
    collectory_url, druid, collectory_api_key, mdata
)
print(new_response)

{'name': '00-dr22810 Rose Test - DR-List-updated-12', 'isPrivate': 'False'}
b'updated DataResource'


In [ ]:
mdata = '{"dataResourceUid": "dr22810", "title": "00-dr22810-Rose Test - DR-List-updated", "description": "", "listType": "TEST", "licence": "CC-BY", "originalFieldList": "['Supplied Name', 'raw.guid', 'raw.Supplied Name', 'raw.family', 'raw.kingdom']", "fieldList": "['rawguid', 'rawSupplied_Name', 'rawfamily', 'rawkingdom']", "facetList": "['rawfamily', 'rawkingdom']", "doi": "", "rowCount": "37", "distinctMatchCount": "37", "authority": "", "category": "", "region": "", "wkt": "", "isVersioned": "", "isAuthoritative": "FALSE", "isPrivate": "TRUE", "isInvasive": "FALSE", "isThreatened": "FALSE", "isBIE": "FALSE", "isSDS": "FALSE", "owner": "56592", "ownerName": "Rosemary OConnor", "lastUpdatedBy": "", "editors": "", "approvedViewers": "", "tags": "", "classification": "", "dateCreated": "1.77129E+12", "metadataLastUpdated": "1.77502E+12", "lastUpdated": "1.76406E+12", "lastUploaded": "1.72498E+12", "private": "FALSE", "bie": "FALSE", "authoritative": "FALSE", "threatened": "FALSE", "invasive": "FALSE", "sds": "FALSE"}'